# H-NBDL: Tables 5–6 — Clinical Pipeline Validation

**All experiments use synthetic data designed to mimic multi-site clinical settings.**
The paper states this transparently. All numbers are real computation.

## Table 5: Multi-Site Radiomics Pipeline
- Synthetic CT radiomic features: 128 dims, J=4 centers, N≈2800, batch effects
- Binary treatment response outcome with site-confounded treatment assignment
- H-NBDL representations → hierarchical logistic regression → cross-site AUC

## Table 6: Cardiac EP Pipeline
- Synthetic EGM features: 256 dims, J=3 labs, N≈1200, motif structure
- Simulated ablation environment → RL agent with dictionary-based states
- Cross-lab transfer evaluation

Runtime: ~30 min on T4

## 0. Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'scipy', 'matplotlib', 'pandas'])

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy import stats
from sklearn.decomposition import DictionaryLearning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.rcParams.update({'font.size': 10, 'figure.dpi': 150})
import matplotlib.pyplot as plt
import json, os, time, copy
from collections import defaultdict

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 1. H-NBDL Model (same as Notebook 1)

In [2]:
class SiteEncoder(nn.Module):
    def __init__(self, D, Kmax, J, hidden=[256,128], Ke=32):
        super().__init__()
        self.site_emb = nn.Embedding(max(J,1), Ke)
        layers = []
        inp = D + Ke
        for h in hidden:
            layers += [nn.Linear(inp,h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.05)]
            inp = h
        self.backbone = nn.Sequential(*layers)
        self.head_mu = nn.Linear(hidden[-1], Kmax)
        self.head_logvar = nn.Linear(hidden[-1], Kmax)
        self.head_logit = nn.Linear(hidden[-1], Kmax)
        nn.init.constant_(self.head_logit.bias, -2.0)
    def forward(self, x, sid):
        h = self.backbone(torch.cat([x, self.site_emb(sid)], -1))
        return self.head_mu(h), self.head_logvar(h), self.head_logit(h)

class HNBDL(nn.Module):
    def __init__(self, D, Kmax=50, J=1, hidden=[256,128]):
        super().__init__()
        self.D_dim, self.Kmax, self.J = D, Kmax, J
        self.encoder = SiteEncoder(D, Kmax, J, hidden)
        self.D_global = nn.Parameter(torch.randn(Kmax, D)*0.05)
        self.D_offsets = nn.Parameter(torch.zeros(max(J,1), Kmax, D))
        self.log_sigma2 = nn.Parameter(torch.tensor(-2.3))
        self.log_tau = nn.Parameter(torch.zeros(Kmax))
        self.log_lambda = nn.Parameter(torch.tensor(2.3))
    @property
    def sigma2(self): return self.log_sigma2.exp().clamp(1e-6,10)
    @property
    def lam(self): return self.log_lambda.exp().clamp(0.01,1000)
    def get_D(self, j): return self.D_global + self.D_offsets[min(j, self.D_offsets.shape[0]-1)]
    def forward(self, x, sid, temp=0.5):
        mu, logvar, logit = self.encoder(x, sid)
        if self.training:
            u = torch.rand_like(logit).clamp(1e-8,1-1e-8)
            g = torch.log(u) - torch.log(1-u)
            z = torch.sigmoid((logit+g)/max(temp,0.01))
        else:
            z = (logit > 0).float()
        s = mu + (0.5*logvar).exp() * torch.randn_like(mu)
        code = z * s
        xhat = torch.zeros_like(x)
        for j in range(self.J):
            mask = (sid == j)
            if mask.any(): xhat[mask] = code[mask] @ self.get_D(j)
        return dict(xhat=xhat, mu=mu, logvar=logvar, logit=logit, z=z, s=s, code=code)
    def compute_loss(self, x, sid, fwd, beta=1.0):
        B = x.shape[0]
        xhat, mu, logvar, logit, z = fwd['xhat'], fwd['mu'], fwd['logvar'], fwd['logit'], fwd['z']
        recon = 0.5/self.sigma2 * ((x-xhat)**2).sum(-1).mean()
        tau = self.log_tau.exp().clamp(0.01,100)
        kl_s = (0.5*z*(logvar.exp()*tau + mu**2*tau - 1 - logvar - tau.log())).sum(-1).mean()
        qz = torch.sigmoid(logit).clamp(1e-6,1-1e-6)
        pp = torch.full_like(qz, 5.0/self.Kmax).clamp(0.01,0.99)
        kl_z = (qz*(qz.log()-pp.log()) + (1-qz)*((1-qz).log()-(1-pp).log())).sum(-1).mean()
        kl_d = (0.5*(self.D_global**2).sum() + 0.5*self.lam*(self.D_offsets**2).sum())/B
        loss = recon + beta*(kl_s + kl_z + kl_d)
        return loss, dict(recon=recon.item(), k_eff=(qz.mean(0)>0.1).sum().item(), sigma2=self.sigma2.item())
    def encode_posterior(self, x, sid):
        self.eval()
        with torch.no_grad():
            mu, logvar, logit = self.encoder(x, sid)
            zp = torch.sigmoid(logit)
            r_mean = zp * mu
            r_var = zp*(logvar.exp()+mu**2) - (zp*mu)**2
        return r_mean, torch.sqrt(r_var.clamp(min=1e-10))

def train_model(X, sids, Kmax=50, J=1, epochs=200, bs=256, lr=1e-3,
                temp_init=1.0, temp_final=0.1, hidden=[256,128], flat=False, verbose=True):
    D = X.shape[1]; aJ = 1 if flat else J
    asids = np.zeros_like(sids) if flat else sids
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    St = torch.tensor(asids, dtype=torch.long, device=DEVICE)
    ds = TensorDataset(Xt, St)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=len(X)>bs)
    model = HNBDL(D, Kmax, aJ, hidden).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    for ep in range(epochs):
        model.train()
        frac = min(ep/max(epochs*0.5,1), 1.0)
        temp = temp_init + (temp_final-temp_init)*frac
        for xb, sb in dl:
            opt.zero_grad()
            fwd = model(xb, sb, temp)
            loss, diag = model.compute_loss(xb, sb, fwd)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sched.step()
        if verbose and (ep%50==0 or ep==epochs-1):
            print(f"  ep {ep:3d}: loss={loss.item():.2f} K_eff={diag['k_eff']:.0f}")
    return model

print('Model and training defined.')

Model and training defined.


## 2. Generate Synthetic Radiomics Data

Mimics multi-site CT radiomics: 128 texture features, J=4 centers with scanner batch effects,
N≈2800, binary treatment response confounded by both features and site.

In [3]:
def generate_radiomics(J=4, Ns=[823, 712, 695, 617], D=128, Kt=20,
                       sigma2=0.5, seed=42):
    """
    Generate synthetic multi-site radiomics data.
    Includes: batch effects, treatment assignment confounded by features,
    binary outcome dependent on a subset of dictionary atoms.
    """
    rng = np.random.default_rng(seed)
    # Dictionary: 12 shared biological atoms + 8 site-specific scanner atoms
    Dg = rng.standard_normal((Kt, D)) * 0.3
    Dg /= np.linalg.norm(Dg, axis=1, keepdims=True)
    pig = np.zeros(Kt)
    pig[:12] = rng.beta(6, 3, 12)  # biological: shared
    pig[12:] = rng.beta(1, 6, 8)   # scanner: site-specific

    # Site-specific batch effects
    site_shifts = [rng.standard_normal(D) * 0.5 for _ in range(J)]
    site_scales = [1.0 + rng.standard_normal(D) * 0.15 for _ in range(J)]
    Ds = [Dg + rng.standard_normal((Kt,D))*0.2 for _ in range(J)]
    pis = []
    for j in range(J):
        pj = rng.beta(np.maximum(8*pig,0.01), np.maximum(8*(1-pig),0.01))
        for k in range(12, Kt):
            pj[k] = rng.beta(6,2) if (k-12)%J==j else rng.beta(1,8)
        pis.append(pj)

    # Outcome model: atoms 0-4 are prognostic, treatment interacts with atoms 2-6
    beta_prog = np.zeros(Kt); beta_prog[:5] = rng.standard_normal(5) * 0.8
    beta_treat = np.zeros(Kt); beta_treat[2:7] = rng.standard_normal(5) * 0.6

    Xl, sl, Tl, yl, Zl, Sl = [], [], [], [], [], []
    for j in range(J):
        for i in range(Ns[j]):
            z = rng.binomial(1, pis[j])
            s = z * rng.standard_normal(Kt)
            x = Ds[j].T @ s + rng.standard_normal(D)*np.sqrt(sigma2)
            # Add batch effects
            x = x * site_scales[j] + site_shifts[j]
            # Treatment: confounded by prognostic atoms
            p_treat = 1/(1+np.exp(-(s @ beta_prog * 0.3 + rng.standard_normal()*0.5)))
            t = rng.binomial(1, np.clip(p_treat, 0.2, 0.8))
            # Outcome: depends on atoms + treatment interaction
            logit_y = s @ beta_prog + t * (s @ beta_treat) + rng.standard_normal()*0.8
            # Site-specific intercept
            logit_y += (j - 1.5) * 0.3
            y = rng.binomial(1, 1/(1+np.exp(-logit_y)))
            Xl.append(x); sl.append(j); Tl.append(t); yl.append(y)
            Zl.append(z); Sl.append(s)

    return dict(X=np.array(Xl), site_ids=np.array(sl), T=np.array(Tl),
                y=np.array(yl), Z_true=np.array(Zl), S_true=np.array(Sl),
                D_global=Dg, D_sites=Ds, Kt=Kt, beta_prog=beta_prog, beta_treat=beta_treat)

rad_data = generate_radiomics()
print(f"Radiomics: N={rad_data['X'].shape[0]}, D={rad_data['X'].shape[1]}, "
      f"sites={[sum(rad_data['site_ids']==j) for j in range(4)]}")
print(f"Treatment rate: {rad_data['T'].mean():.2f}, outcome rate: {rad_data['y'].mean():.2f}")

Radiomics: N=2847, D=128, sites=[np.int64(823), np.int64(712), np.int64(695), np.int64(617)]
Treatment rate: 0.50, outcome rate: 0.49


## 3. Table 5: Radiomics Benchmark

In [4]:
def get_representations(X, site_ids, method, J=4, seed=0):
    """Get representations from different methods."""
    N, D = X.shape
    if method == 'Raw':
        return X, None
    elif method == 'ComBat':
        # Simple location-scale batch correction per site
        X_corrected = X.copy()
        global_mean = X.mean(0)
        global_std = X.std(0) + 1e-8
        for j in range(J):
            mask = site_ids == j
            site_mean = X[mask].mean(0)
            site_std = X[mask].std(0) + 1e-8
            X_corrected[mask] = (X[mask] - site_mean) / site_std * global_std + global_mean
        return X_corrected, None
    elif method.startswith('K-SVD'):
        K = int(method.split('=')[1].rstrip(')'))
        dl = DictionaryLearning(n_components=K, alpha=1.0, max_iter=200,
                                transform_algorithm='omp', random_state=seed)
        codes = dl.fit_transform(X)
        return codes, None
    elif method.startswith('VAE'):
        K = int(method.split('=')[1].rstrip(')'))
        model = train_model(X, site_ids, Kmax=K, J=1, epochs=150, flat=True, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St = torch.zeros(N, dtype=torch.long, device=DEVICE)
        rm, rs = model.encode_posterior(Xt, St)
        return rm.cpu().numpy(), rs.cpu().numpy()
    elif method == 'BDL-MAP':
        rng = np.random.default_rng(seed)
        K = 20
        _, sv, Vt = np.linalg.svd(X, full_matrices=False)
        De = Vt[:K]; De /= np.linalg.norm(De,axis=1,keepdims=True)+1e-10
        ard = np.ones(K); s2 = float(np.var(X))*0.5
        for _ in range(200):
            reg = np.diag(ard)*s2; S = np.linalg.solve(De@De.T+reg, De@X.T).T
            De = np.linalg.lstsq(S, X, rcond=None)[0]
            nm = np.linalg.norm(De,axis=1,keepdims=True); nm[nm<1e-10]=1; De /= nm
            ard = D/(np.sum(De**2,axis=1)+1e-10); s2 = max(float(np.mean((X-S@De)**2)),1e-6)
        Sv = s2 * np.diag(np.linalg.inv(De@De.T + np.diag(ard)*s2))
        return S, np.sqrt(np.tile(Sv, (N,1))+1e-8)
    elif method == 'Flat NBDL':
        model = train_model(X, site_ids, Kmax=50, J=J, epochs=150, flat=True, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St = torch.zeros(N, dtype=torch.long, device=DEVICE)
        rm, rs = model.encode_posterior(Xt, St)
        return rm.cpu().numpy(), rs.cpu().numpy()
    elif method == 'H-NBDL':
        model = train_model(X, site_ids, Kmax=50, J=J, epochs=200, flat=False, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St = torch.tensor(site_ids, dtype=torch.long, device=DEVICE)
        rm, rs = model.encode_posterior(Xt, St)
        return rm.cpu().numpy(), rs.cpu().numpy()
    else:
        raise ValueError(f'Unknown method: {method}')

def evaluate_radiomics(X, site_ids, T, y, method, J=4, n_folds=5, seed=0):
    """5-fold stratified CV for treatment response prediction."""
    reps, _ = get_representations(X, site_ids, method, J, seed)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    aucs = []
    site_aucs = {j: [] for j in range(J)}
    for train_idx, test_idx in skf.split(reps, y):
        scaler = StandardScaler().fit(reps[train_idx])
        X_train = scaler.transform(reps[train_idx])
        X_test = scaler.transform(reps[test_idx])
        # Add treatment as feature
        X_train_full = np.column_stack([X_train, T[train_idx]])
        X_test_full = np.column_stack([X_test, T[test_idx]])
        clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        clf.fit(X_train_full, y[train_idx])
        y_prob = clf.predict_proba(X_test_full)[:, 1]
        try:
            aucs.append(roc_auc_score(y[test_idx], y_prob))
        except ValueError:
            aucs.append(0.5)
        # Per-site AUC
        for j in range(J):
            site_mask = site_ids[test_idx] == j
            if site_mask.sum() > 5 and len(np.unique(y[test_idx][site_mask])) > 1:
                site_aucs[j].append(roc_auc_score(y[test_idx][site_mask], y_prob[site_mask]))
    overall_auc = np.mean(aucs)
    min_site_auc = min(np.mean(v) if v else 0.5 for v in site_aucs.values())
    return overall_auc, np.std(aucs), min_site_auc

# ── Run Table 5 ──
print('='*70)
print('TABLE 5: Multi-Site Radiomics (synthetic clinical data)')
print('='*70)
d = rad_data
methods_t5 = ['Raw', 'ComBat', 'K-SVD(K=20)', 'VAE(d=20)', 'BDL-MAP', 'Flat NBDL', 'H-NBDL']
t5 = {}
for method in methods_t5:
    print(f'  {method}...')
    auc, auc_std, min_auc = evaluate_radiomics(d['X'], d['site_ids'], d['T'], d['y'],
                                                method, J=4)
    t5[method] = dict(auc=round(auc,3), auc_std=round(auc_std,3), min_site_auc=round(min_auc,3))
    print(f'    AUC={auc:.3f}±{auc_std:.3f}, min-site={min_auc:.3f}')

print('\n' + '='*70)
print(f'{"Method":<16s} {"AUC ↑":>12s} {"Min-site AUC ↑":>16s}')
print('-'*50)
for m, v in t5.items():
    print(f'{m:<16s} {v["auc"]:.3f}±{v["auc_std"]:.3f}   {v["min_site_auc"]:.3f}')
print('='*70)

with open('results/table5_radiomics.json', 'w') as f: json.dump(t5, f, indent=2)
print('Saved: results/table5_radiomics.json')

TABLE 5: Multi-Site Radiomics (synthetic clinical data)
  Raw...
    AUC=0.659±0.031, min-site=0.600
  ComBat...
    AUC=0.630±0.024, min-site=0.603
  K-SVD(K=20)...
    AUC=0.619±0.030, min-site=0.514
  VAE(d=20)...
    AUC=0.634±0.029, min-site=0.562
  BDL-MAP...
    AUC=0.632±0.029, min-site=0.568
  Flat NBDL...
    AUC=0.670±0.031, min-site=0.623
  H-NBDL...
    AUC=0.607±0.016, min-site=0.576

Method                  AUC ↑   Min-site AUC ↑
--------------------------------------------------
Raw              0.659±0.031   0.600
ComBat           0.630±0.024   0.603
K-SVD(K=20)      0.619±0.030   0.514
VAE(d=20)        0.634±0.029   0.562
BDL-MAP          0.632±0.029   0.568
Flat NBDL        0.670±0.031   0.623
H-NBDL           0.607±0.016   0.576
Saved: results/table5_radiomics.json


## 4. Generate Synthetic EP Data + RL Environment

Synthetic cardiac electrophysiology: 256-dim EGM features with motif structure,
J=3 labs. Simulated ablation environment where the RL agent selects ablation
targets based on dictionary representations.

In [5]:
def generate_ep_data(J=3, Ns=[450, 400, 380], D=256, Kt=18, sigma2=0.3, seed=42):
    """
    Generate synthetic EP data: EGM features with motif structure.
    11 shared motifs + 7 lab-specific motifs (catheter artifacts, mapping density).
    """
    rng = np.random.default_rng(seed)
    # Motifs: structured temporal patterns
    Dg = np.zeros((Kt, D))
    for k in range(Kt):
        # Each motif is a localized pattern (Gaussian bumps)
        center = rng.integers(20, D-20)
        width = rng.integers(10, 40)
        t = np.arange(D)
        Dg[k] = np.exp(-0.5*((t-center)/width)**2) * rng.choice([-1,1])
        # Add some oscillatory structure
        Dg[k] += 0.3 * np.sin(2*np.pi*rng.uniform(0.01,0.05)*t + rng.uniform(0,2*np.pi))
    Dg /= np.linalg.norm(Dg, axis=1, keepdims=True)

    pig = np.zeros(Kt)
    pig[:11] = rng.beta(6, 3, 11)  # shared motifs
    pig[11:] = rng.beta(1, 6, 7)   # lab-specific

    Ds = [Dg + rng.standard_normal((Kt,D))*0.15 for _ in range(J)]
    pis = []
    for j in range(J):
        pj = rng.beta(np.maximum(8*pig,0.01), np.maximum(8*(1-pig),0.01))
        for k in range(11, Kt):
            pj[k] = rng.beta(6,2) if (k-11)%J==j else rng.beta(1,8)
        pis.append(pj)

    # Ablation success depends on correctly identifying motifs 0-5
    ablation_weights = np.zeros(Kt)
    ablation_weights[:6] = rng.uniform(0.5, 1.5, 6)

    Xl, sl, Zl, Sl = [], [], [], []
    for j in range(J):
        for i in range(Ns[j]):
            z = rng.binomial(1, pis[j])
            s = z * rng.standard_normal(Kt)
            x = Ds[j].T @ s + rng.standard_normal(D)*np.sqrt(sigma2)
            Xl.append(x); sl.append(j); Zl.append(z); Sl.append(s)

    return dict(X=np.array(Xl), site_ids=np.array(sl), Z_true=np.array(Zl),
                S_true=np.array(Sl), D_global=Dg, Kt=Kt,
                ablation_weights=ablation_weights, n_shared=11, n_specific=7)

class AblationEnv:
    """
    Simulated ablation environment.
    State: patient representation (from dictionary model).
    Action: select ablation target region (discrete, K_max choices).
    Reward: based on whether the action targets an active pathological motif.
    """
    def __init__(self, representations, Z_true, ablation_weights, rng=None):
        self.reps = representations
        self.Z = Z_true
        self.weights = ablation_weights
        self.N = representations.shape[0]
        self.rng = rng or np.random.default_rng(0)
        self.idx = 0

    def reset(self):
        self.idx = self.rng.integers(0, self.N)
        return self.reps[self.idx]

    def step(self, action):
        """Action is an index into K atoms. Reward if targeting active pathological atom."""
        z = self.Z[self.idx]
        K = len(self.weights)
        action = min(action, K-1)
        # Reward: high if atom is active AND has high ablation weight
        if action < len(z) and z[action] > 0 and self.weights[action] > 0:
            reward = float(self.weights[action] * z[action])
            success = reward > 0.3
        else:
            reward = -0.1
            success = False
        self.idx = self.rng.integers(0, self.N)
        return self.reps[self.idx], reward, success

ep_data = generate_ep_data()
print(f"EP: N={ep_data['X'].shape[0]}, D={ep_data['X'].shape[1]}, "
      f"labs={[sum(ep_data['site_ids']==j) for j in range(3)]}")
print(f"Motifs: {ep_data['n_shared']} shared, {ep_data['n_specific']} lab-specific")

EP: N=1230, D=256, labs=[np.int64(450), np.int64(400), np.int64(380)]
Motifs: 11 shared, 7 lab-specific


## 5. Table 6: EP + RL Benchmark

In [6]:
class SimpleRLAgent:
    """Epsilon-greedy Q-learning agent for ablation target selection."""
    def __init__(self, state_dim, n_actions, hidden=128, lr=1e-3):
        self.n_actions = n_actions
        self.q_net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        ).to(DEVICE)
        self.opt = torch.optim.Adam(self.q_net.parameters(), lr=lr)
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.05
        self.rng = np.random.default_rng(0)

    def act(self, state):
        if self.rng.random() < self.epsilon:
            return self.rng.integers(0, self.n_actions)
        with torch.no_grad():
            st = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            return self.q_net(st).argmax(-1).item()

    def update(self, state, action, reward, next_state, gamma=0.95):
        st = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        nst = torch.tensor(next_state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        q = self.q_net(st)[0, action]
        with torch.no_grad():
            q_next = self.q_net(nst).max(-1).values[0]
        target = reward + gamma * q_next
        loss = F.mse_loss(q, torch.tensor(target, device=DEVICE))
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)

def run_rl_experiment(reps, Z_true, ablation_weights, n_episodes=5000,
                     n_actions=18, eval_episodes=500, seed=0):
    """Train RL agent and evaluate ablation success rate."""
    rng = np.random.default_rng(seed)
    state_dim = reps.shape[1]
    env = AblationEnv(reps, Z_true, ablation_weights, rng)
    agent = SimpleRLAgent(state_dim, n_actions)

    # Train
    for ep in range(n_episodes):
        state = env.reset()
        action = agent.act(state)
        next_state, reward, success = env.step(action)
        agent.update(state, action, reward, next_state)

    # Evaluate
    agent.epsilon = 0.0  # greedy
    successes = 0
    for _ in range(eval_episodes):
        state = env.reset()
        action = agent.act(state)
        _, _, success = env.step(action)
        successes += int(success)
    return successes / eval_episodes

def run_rl_transfer(reps_train, Z_train, reps_test, Z_test, ablation_weights,
                    n_actions=18, seed=0):
    """Train on one set, evaluate on another (cross-lab transfer)."""
    rng = np.random.default_rng(seed)
    state_dim = reps_train.shape[1]
    env_train = AblationEnv(reps_train, Z_train, ablation_weights, rng)
    agent = SimpleRLAgent(state_dim, n_actions)

    for ep in range(5000):
        state = env_train.reset()
        action = agent.act(state)
        next_state, reward, _ = env_train.step(action)
        agent.update(state, action, reward, next_state)

    # Transfer eval
    agent.epsilon = 0.0
    env_test = AblationEnv(reps_test, Z_test, ablation_weights, rng)
    successes = 0
    for _ in range(500):
        state = env_test.reset()
        action = agent.act(state)
        _, _, success = env_test.step(action)
        successes += int(success)
    return successes / 500

# ── Run Table 6 ──
print('='*70)
print('TABLE 6: Cardiac EP + RL (synthetic clinical data)')
print('='*70)

d = ep_data
X, sids, Zt, St = d['X'], d['site_ids'], d['Z_true'], d['S_true']
aw = d['ablation_weights']; Kt = d['Kt']

methods_t6 = ['Raw', 'K-SVD(K=20)', 'VAE(d=30)', 'BDL-MAP', 'Flat NBDL', 'H-NBDL']
t6 = {}

for method in methods_t6:
    print(f'  {method}...')
    if method == 'Raw':
        reps = X
    elif method == 'K-SVD(K=20)':
        dl = DictionaryLearning(n_components=20, alpha=1., max_iter=200, random_state=0)
        reps = dl.fit_transform(X)
    elif method == 'VAE(d=30)':
        model = train_model(X, sids, Kmax=30, J=1, epochs=150, flat=True, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St0 = torch.zeros(len(X), dtype=torch.long, device=DEVICE)
        rm, _ = model.encode_posterior(Xt, St0); reps = rm.cpu().numpy()
    elif method == 'BDL-MAP':
        _, sv, Vt = np.linalg.svd(X, full_matrices=False)
        De = Vt[:20]; De /= np.linalg.norm(De,axis=1,keepdims=True)+1e-10
        ard = np.ones(20); s2 = float(np.var(X))*0.5
        for _ in range(200):
            reg = np.diag(ard)*s2; S = np.linalg.solve(De@De.T+reg, De@X.T).T
            De = np.linalg.lstsq(S, X, rcond=None)[0]
            nm = np.linalg.norm(De,axis=1,keepdims=True); nm[nm<1e-10]=1; De /= nm
            ard = 256/(np.sum(De**2,axis=1)+1e-10); s2 = max(float(np.mean((X-S@De)**2)),1e-6)
        reps = S
    elif method == 'Flat NBDL':
        model = train_model(X, sids, Kmax=50, J=3, epochs=150, flat=True, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St0 = torch.zeros(len(X), dtype=torch.long, device=DEVICE)
        rm, _ = model.encode_posterior(Xt, St0); reps = rm.cpu().numpy()
    elif method == 'H-NBDL':
        model = train_model(X, sids, Kmax=50, J=3, epochs=200, flat=False, verbose=False)
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        St_real = torch.tensor(sids, dtype=torch.long, device=DEVICE)
        rm, _ = model.encode_posterior(Xt, St_real); reps = rm.cpu().numpy()

    # Ablation success
    success = run_rl_experiment(reps, Zt, aw, n_actions=Kt, seed=0)

    # Cross-lab transfer: train on labs 0+1, test on lab 2
    train_mask = sids != 2; test_mask = sids == 2
    transfer = run_rl_transfer(reps[train_mask], Zt[train_mask],
                                reps[test_mask], Zt[test_mask], aw, n_actions=Kt)

    t6[method] = dict(success=round(success,3), transfer=round(transfer,3))
    print(f'    Success={success:.3f}, Transfer={transfer:.3f}')

print('\n' + '='*70)
print(f'{"Method":<16s} {"Ablation Success ↑":>20s} {"Transfer ↑":>14s}')
print('-'*55)
for m, v in t6.items():
    print(f'{m:<16s} {v["success"]:.3f}             {v["transfer"]:.3f}')
print('='*70)

with open('results/table6_ep.json', 'w') as f: json.dump(t6, f, indent=2)
print('Saved: results/table6_ep.json')

TABLE 6: Cardiac EP + RL (synthetic clinical data)
  Raw...


/tmp/ipykernel_2948/1545427203.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss = F.mse_loss(q, torch.tensor(target, device=DEVICE))


    Success=0.780, Transfer=0.756
  K-SVD(K=20)...


/tmp/ipykernel_2948/1545427203.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss = F.mse_loss(q, torch.tensor(target, device=DEVICE))


    Success=0.638, Transfer=0.756
  VAE(d=30)...
    Success=0.604, Transfer=0.756
  BDL-MAP...
    Success=0.780, Transfer=0.326
  Flat NBDL...
    Success=0.780, Transfer=0.684
  H-NBDL...
    Success=0.646, Transfer=0.756

Method             Ablation Success ↑     Transfer ↑
-------------------------------------------------------
Raw              0.780             0.756
K-SVD(K=20)      0.638             0.756
VAE(d=30)        0.604             0.756
BDL-MAP          0.780             0.326
Flat NBDL        0.780             0.684
H-NBDL           0.646             0.756
Saved: results/table6_ep.json


## 6. Summary: All Numbers for Tables 5-6

In [7]:
print('='*70)
print('NUMBERS FOR PAPER — TABLES 5 AND 6')
print('All from synthetic clinical data. All real computation.')
print('='*70)

t5 = json.load(open('results/table5_radiomics.json'))
print('\n── TABLE 5: Radiomics (N=2847, D=128, J=4 centers) ──')
print(f'{"Method":<16s} {"AUC":>10s} {"Min-site":>10s}')
for m, v in t5.items():
    print(f'{m:<16s} {v["auc"]:.3f}±{v["auc_std"]:.3f}   {v["min_site_auc"]:.3f}')

t6 = json.load(open('results/table6_ep.json'))
print('\n── TABLE 6: Cardiac EP (N=1230, D=256, J=3 labs) ──')
print(f'{"Method":<16s} {"Success":>10s} {"Transfer":>10s}')
for m, v in t6.items():
    print(f'{m:<16s} {v["success"]:.3f}      {v["transfer"]:.3f}')

print('\n' + '='*70)
print('ALL NUMBERS ARE GENUINE COMPUTATION ON SYNTHETIC CLINICAL DATA.')
print('='*70)

NUMBERS FOR PAPER — TABLES 5 AND 6
All from synthetic clinical data. All real computation.

── TABLE 5: Radiomics (N=2847, D=128, J=4 centers) ──
Method                  AUC   Min-site
Raw              0.659±0.031   0.600
ComBat           0.630±0.024   0.603
K-SVD(K=20)      0.619±0.030   0.514
VAE(d=20)        0.634±0.029   0.562
BDL-MAP          0.632±0.029   0.568
Flat NBDL        0.670±0.031   0.623
H-NBDL           0.607±0.016   0.576

── TABLE 6: Cardiac EP (N=1230, D=256, J=3 labs) ──
Method              Success   Transfer
Raw              0.780      0.756
K-SVD(K=20)      0.638      0.756
VAE(d=30)        0.604      0.756
BDL-MAP          0.780      0.326
Flat NBDL        0.780      0.684
H-NBDL           0.646      0.756

ALL NUMBERS ARE GENUINE COMPUTATION ON SYNTHETIC CLINICAL DATA.


## 7. Download

In [8]:
import shutil
shutil.make_archive('tables56_results', 'zip', '.', 'results')
try:
    from google.colab import files
    files.download('tables56_results.zip')
except:
    print('Files saved locally in results/')
print('Done.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
